# Taxonomy ergonomics

Phase-0 prerequisites for flows: hashable maps, bulk construction, a polars
frame view, and `Groups` from `group_by`.

In [ ]:
from summer4 import Groups, Property, PropertyMap

state = Property("state", ("S", "I", "R"))
age = Property("age", ("0-4", "5-9", "10+"))

pm = PropertyMap.from_properties([state, age])
assert len(pm) == 9
assert pm == PropertyMap.from_property(state).stratify(age)

In [ ]:
again = PropertyMap.from_properties([state, age])
assert pm is not again
assert pm == again
assert hash(pm) == hash(again)
assert {pm: "cached"}[again] == "cached"

In [ ]:
frame = pm.to_frame()
assert frame.columns == ["state", "age"]
assert frame.height == len(pm)
assert set(frame["state"].to_list()) == {"S", "I", "R"}
frame.head(3)

In [ ]:
groups = pm.group_by(state, age)
assert isinstance(groups, Groups)
assert len(groups) == 9
key = (state["I"], age["0-4"])
assert groups[key].tolist() == pm.select(state["I"] & age["0-4"]).tolist()
assert sum(idx.size for idx in groups.values()) == len(pm)